# Image Preprocessing


In [ ]:
import os

import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

IMG_SIZE = 64


def preprocess_images(df, test_size=0.15, val_size=0.15, random_state=42):
    images, labels = [], []

    for _, row in df.iterrows():
        path = row["file_path"]
        if os.path.exists(path):
            try:
                img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
                images.append(np.asarray(img, dtype=np.float32) / 255.0)
                labels.append(row["label"])
            except Exception:
                continue

    X = np.array(images, dtype=np.float32)
    y = np.array(labels, dtype=np.float32)

    holdout = test_size + val_size
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=holdout, random_state=random_state, stratify=y
    )

    val_ratio = val_size / holdout
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=(1.0 - val_ratio), random_state=random_state, stratify=y_temp
    )

    print(f"Loaded {len(X)} images. Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)